# Lekcija 03 - Agentni oblikovalski vzorci

V tej lekciji bomo raziskali tri temeljne oblikovalske vzorce za gradnjo učinkovitih AI agentov:

1. **Jasna navodila za agenta** — Oblikovanje natančnih, vlog določajočih napotkov, ki usmerjajo vedenje agenta  
2. **Strukturiran izhod s Pydantic modeli** — Zagotavljanje, da agenti vračajo predvidljive, preverjene podatke  
3. **Agent z eno odgovornostjo** — Oblikovanje fokusiranih agentov, ki vsak opravi eno stvar dobro  

Vsak vzorec bomo uporabili v primeru **priporočevalca potovalnih destinacij**, z postopnim gradnjim sistema, ki lahko predlaga destinacije, preverja razpoložljivost in ureja logistiko.


## Namestitev


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Vzorec 1: Jasna navodila za agenta

Najbolj učinkovit vzorec je tudi najpreprostejši: pisanje jasnih, podrobnih navodil za vašega agenta.

Dobra navodila opredeljujejo:
- **Kdo** je agent (osebnost in ton)
- **Kaj** naj počne (odgovornosti korak za korakom)
- **Kako** naj se obnaša (omejitve in slog)

Spodaj ustvarjamo potovalnega koncierge agenta z izrecnimi navodili, ki oblikujejo vsak njegov odgovor.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Vzorec 2: Strukturiran izhod z modeli Pydantic

Brezstrukturiran tekst je uporaben za pogovor, vendar sistemi v nadaljnjih fazah potrebujejo strukturirane podatke.  
Z združitvijo **modelov Pydantic** s **pomožno funkcijo** lahko:

- Določimo natančno shemo za izhod agenta  
- Samodejno preverimo odgovore  
- Zanesljivo vključimo rezultate agenta v logiko aplikacije  

Uvajamo tudi pripomoček, ki vrne podrobnosti cilja, tako da agent podlaga svoja priporočila v resničnih podatkih.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Vzorec 3: Agent z eno samo odgovornostjo

Zahtevnejše naloge imajo koristi od razdelitve dela med več osredotočenih agentov, od katerih ima vsak eno samo odgovornost:

- **Strokovnjak za destinacijo**, ki pozna kraje in razpoložljivost
- **Načrtovalec logistike**, ki ureja lete, hotele in itinerarje

To odseva načelo programske opreme *ločevanja skrbi* — vsak agent je lažje testirati, vzdrževati in izboljševati neodvisno.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Povzetek

V tej lekciji smo tri oblikovne vzorce agentov uporabili v scenariju priporočila potovanj:

| Vzorec | Glavna ideja | Korist |
|---|---|---|
| **Jasna navodila** | Določite persono, odgovornosti in omejitve vnaprej | Dosledno vedenje agenta v skladu z blagovno znamko |
| **Strukturiran izhod** | Uporabite Pydantic modele kot format odgovora | Validirani, za stroj berljivi rezultati |
| **Enotna odgovornost** | Dodelite vsakemu agentu eno osredotočeno nalogo | Lažje testiranje, vzdrževanje in sestavljanje |

Ti vzorci se naravno sestavljajo — lahko združite jasna navodila s strukturiranim izhodom znotraj agenta z eno odgovornostjo za izdelavo robustnih, proizvodnji pripravljenih sistemov.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas opozarjamo, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvornega jeziku je treba šteti za avtoritativni vir. Za pomembne informacije priporočamo strokovni človeški prevod. Nismo odgovorni za morebitna nesporazume ali napačne razlage, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
